# Spatial Data Science with CityJSON (No Internet)

<div class="alert alert-block alert-info""><b>This is a stand-alone Notebook for use in low resource settings. NO internet connection is necessary.</b> 

<br>
<br>

While an internet connection is **NOT** necessary you will **NEED** to have sourced an **[osm.pbf](https://wiki.openstreetmap.org/wiki/PBF_Format).**

If you would like to do the entire Notebook and estimate the **Annual Average Solar Potential, per home** you will **NEED** to have sourced a **ghi.tif**.  
A **ghi.tif** raster comes with the **LTAy_YearlySum.GeoTiff** dataset.  
These can be harvested from the **[Global Solar Atlas download section](https://globalsolaratlas.info/download/) and are available for each country on Earth.**
</div>

The purpose of this Notebook is to ***work with*** the product of [osm_LoD1_3DCityModel](https://github.com/AdrianKriger/osm_LoD1_3DCityModel); a previously created CityJSON city model.

<div class="alert alert-block alert-warning"><b>This notebook will:</b>

> **1. allow the user to execute an application of Spatial Data Science**  
>
>> **a)  [population estimation](#Section1a)** _--with a previous census metric population growth rate and projected (future) population are also possible_  **and**  
>> **b)  a measure of [Building Volume per Capita](#Section1b)**
>
> **2. further applications of Spatial Data Science**  
>
>> **i.** calculate percentage **homes and population with direct access to on-site renewable energy infrastructure** *--rooftop photovoltaic panels (PV) and solar water heaters (SWH).*  
>> **ii.** estimate the [**Annual Average Solar** *(photovoltaic)* **Potential**](https://www.worldbank.org/en/topic/energy/publication/solar-photovoltaic-power-potential-by-country), per home.
>
> **3. produce [an interactive visualization](#Section1b)** *- which a user can navigate, query and share* **that**;
> > **a) [colour buildings by type](#Section2a)** *(to easily visualize building stock)* 
>
> **4. propose several [Geography and Sustainable Development Education *conversation starters*](#Section3) for Secondary and Tertiary level students**
</div>

In [1]:
#load the magic

%matplotlib inline
import os
import sys
from pathlib import Path
import webbrowser

import numpy as np
import pandas as pd
import shapely
from shapely.geometry import Polygon, shape, mapping
import json
import geojson

#import city3D
#from cjio import cityjson

from osgeo import gdal, osr

import matplotlib.pyplot as plt
#import pydeck as pdk

In [2]:
#- get current working directory (notebook location)
current_dir = os.getcwd()

#- go one level up
parent_dir = os.path.dirname(current_dir)
# Add parent directory to sys.path if needed
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

#- import
import city3D

from cjio import cityjson

<div class="alert alert-block alert-info"><b></b> 
    
**PLEASE SUPPLY YOUR OWN [.mbtiles file](https://wiki.openstreetmap.org/wiki/PBF_Format)**  

***Either [crop an area](https://extract.bbbike.org/?format=mbtiles-openmaptiles.zip)***, or select a predefined area **[from any number of providers](https://wiki.openstreetmap.org/wiki/MBTiles)**
</div>

In [3]:
#- works fine
import warnings
warnings.filterwarnings('ignore')

**The area under investigation is [Woodstock, Cape Town. South Africa](https://en.wikipedia.org/wiki/Woodstock,_Cape_Town).**

In [4]:
#- change to harvest the appropriate CityJSON

#jparams = json.load(open('../param/wStock_param.json'))       
#jparams = json.load(open('../param/uEstate_param.json'))   
jparams = json.load(open('../param/sRiver_param.json'))   
#jparams = json.load(open('../param/saao_param.json'))       

In [5]:
cm = cityjson.load(path=jparams['cjsn_solid']) #-- citjsnClean_uEstate10m.json in the result folder

In [6]:
print(cm)

CityJSON version = 1.1
EPSG = 32734
bbox = [264142.0606272015, 6240967.812376269, 0.03999999910593033, 267325.9684924788, 6244521.576538833, 132.14]
=== CityObjects ===
|-- TINRelief (1)
|-- Building (1377)
materials = False
textures = False


In [7]:
df = cm.to_dataframe()
#- remove the first feature: the terrain
df = df[1:]    

#- harvest the crs
theinfo = cm.get_info()
crs = theinfo[1]

# account for holes
def coords_to_polygon(rings):
    outer = rings[0]                              # first ring is the shell
    holes = rings[1:] if len(rings) > 1 else None
    return Polygon(shell=outer, holes=holes)

# Convert JSON string to Python list
df['footprint_coords_list'] = df['footprint'].apply(json.loads)

# create home-baked gdf
#gdf = gpd.GeoDataFrame(df, geometry=df['footprint_coords_list'].apply(coords_to_polygon), crs=crs[7:])#jparams['crs'])
gdf = city3D.GeoDataFrameLite(df)
gdf['geometry'] = gdf['footprint_coords_list'].apply(coords_to_polygon)
gdf.crs = crs[7:]
# Drop columns inplace
gdf.drop(columns=['footprint', 'footprint_coords_list'], inplace=True)

In [8]:
gdf.head(2)

,id,osm_id,address,building,building:levels,building_height,roof_height,ground_height,plus_code,operator,building:flats,amenity,social_facility,bottom_roof_height,beds,residential,building:use,rooms,building:units,geometry
6383946,0,6383946,Western Cape Metrorail - Infrastructure Buildi...,office,1.0,4.1,8.21,4.110000,4FRW3FCF+946,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((266353.2773689979 6242850.21272516, ..."
13328172,1,13328172,NaN,school,2.0,6.9,23.93,17.030001,4FRW3F96+6MG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((265366.0837814461 6242509.526659243,..."


<div class="alert alert-block alert-info"><b>Craveat</b> 
    
This community *(Woodstock, Salt River and Observatory)* is the second oldest community in South Africa. The buildings are old. Many have been repurposed. To account for refurbishment *--be as representative as possible--* and conform to the **[OpenStreetMap Guide](https://wiki.openstreetmap.org/wiki/Beginners%27_guide)** we typically tag these:  
`building=*` *~ the original purpose* `+` `building:use=*` *~ the current use*.

Furthermore; tagging in this community identifies **social housing, social facilities** (care home, shelter, etc.) **and informal housing** (backyard dwelling, shack, etc.) as `building / :use=residential`. **Student accomodation** includes the `residential=student` tag
</div>

In [9]:
#- to account for idiosyncratic mapping: replace building= (old function) if building:use= (new purpose) is present
gdf2 = gdf.copy()

#- 1
gdf2.loc[
    # The condition to find rows where 'building:use' is 'residential'
    # This check ensures the column exists, preventing a KeyError
    (gdf2['building:use'] == 'residential') & ~gdf2['building:use'].isna() 
    if 'building:use' in gdf2.columns else [False] * len(gdf2), 
    
    # The column to be updated
    'building'] = (
    # The value to assign to the 'building' column
    gdf2['building:use'] 
    if 'building:use' in gdf2.columns else None
)

#- 2
gdf2.loc[
    # The condition to find rows where 'building:use' is 'residential'
    # This check ensures the column exists, preventing a KeyError
    (gdf2['residential'] == 'student') & ~gdf2['residential'].isna() 
    if 'residential' in gdf2.columns else [False] * len(gdf2), 
    
    # The column to be updated
    'building'] = (
    # The value to assign to the 'building' column
    gdf2['residential'] 
    if 'residential' in gdf2.columns else None
)

## 1. Spatial Data Science

<div class="alert alert-block alert-warning"><b>We start with basic spatial analysis</b>  
    
     
- We'll [estimate the population](#Section1a), within our area of interest, and then  
- calculate the [Building Volume Per Capita (BVPC)](#Section1b).
</div>

While estimating population is well documented; recent investigations to **understand overcrowding** have led to newer measurements.  

The most noteable of these is **Building Volume Per Capita (BVPC)** [(Ghosh, T; et al. 2020)](https://www.researchgate.net/publication/343185735_Building_Volume_Per_Capita_BVPC_A_Spatially_Explicit_Measure_of_Inequality_Relevant_to_the_SDGs). BVPC is the cubic meters of building per person. **BVPC tells us how much space one person has per residential living unit** (a house / apartment / etc.). It is ***a proxy measure of economic inequality and a direct measure of housing inequality***.

BVPC builds on the work of [(Reddy, A and Leslie, T.F., 2013)](https://www.tandfonline.com/doi/abs/10.1080/02723638.2015.1060696?journalCode=rurb20) and attempts to integrate with several **[Sustainable Development Goals](https://sdgs.un.org/goals)** (most noteably: **[SDG 11: Developing sustainable cities and communities](https://sdgs.un.org/goals/goal11)**) and captures the average ***'living space'*** each person has in their home.

<div class="alert alert-block alert-info"><b>These analysis expect the user to have some basic knowledge about the environment under inquiry / investigation</b> </div>

In [10]:
#gdf.head(2)
len(gdf2)

1377

In [11]:
#gdf.plot()
# have a look at the building type and amenities available
gdf2['building'].unique()

array(['office', 'school', 'yes', 'apartments', 'industrial',
       'commercial', 'train_station', 'retail', 'garage', 'workshop',
       'church', 'roof', 'house', 'transportation', 'fire_station',
       'pavilion', 'college', 'dormitory', 'university', 'clubhouse',
       'student', 'warehouse', 'civic', 'parking', 'residential',
       'cinema', 'construction', 'manufacture', 'kindergarten', 'mosque',
       'hall', 'carport', 'terraced', 'guardhouse', 'terrace', 'silo',
       'storage_tank'], dtype=object)

<a id='Section1a'></a>

<div class="alert alert-block alert-success"><b>1.  a) Estimate Population:</b> 
    
_(with population growth rate and population projection possible too)_ </div>

In [12]:
#- some data wrangling
with pd.option_context("future.no_silent_downcasting", True):
    gdf2 = gdf2.assign(**{
        col: pd.to_numeric(
            gdf2[col].fillna(0).infer_objects(copy=False), errors='coerce'
        )
        for col in ['building:flats', 'building:units', 'beds', 'rooms', 'building:levels']
        if col in gdf2.columns
    })

gdf2.head(2)

,id,osm_id,address,building,building:levels,building_height,roof_height,ground_height,plus_code,operator,building:flats,amenity,social_facility,bottom_roof_height,beds,residential,building:use,rooms,building:units,geometry
6383946,0,6383946,Western Cape Metrorail - Infrastructure Buildi...,office,1.0,4.1,8.21,4.110000,4FRW3FCF+946,NaN,0,NaN,NaN,NaN,0,NaN,NaN,0,0,"POLYGON ((266353.2773689979 6242850.21272516, ..."
13328172,1,13328172,NaN,school,2.0,6.9,23.93,17.030001,4FRW3F96+6MG,NaN,0,NaN,NaN,NaN,0,NaN,NaN,0,0,"POLYGON ((265366.0837814461 6242509.526659243,..."


In [13]:
#gdf2['building'].value_counts()

In [14]:
#--we only want building=house or =apartment or =residential
gdf_pop = gdf2[gdf2["building"].isin(['house', 'semidetached_house', 'terrace', 'terraced', 'apartments', 'residential', 'dormitory', 'cabin', 'student', 'garage'])].copy()

In [15]:
#inf = gdf_pop[gdf_pop["building"].isin(['residential'])].copy()
len(gdf_pop)

999

In [16]:
#gdf_pop.plot()
gdf_pop['building'].value_counts()

building
house          948
apartments      23
dormitory       10
garage           6
student          6
terraced         4
residential      1
terrace          1
Name: count, dtype: int64

**This area is urban with single and 2-storey level housing units. To estimate population is thus pretty straight forward.**

<div class="alert alert-block alert-info"><b>We start with local knowledge.</b></div>

**On average there are roughly `4` people per `building:house` in this area.**  

**[social housing](https://en.wikipedia.org/wiki/Public_housing)** is tagged `building:residential` with the `3` people per building or `building:flats * 3` if the building is an *apartment-type* complex

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    

We will execute the calculation programmatically. **Fill in the relevant variables in the _`cell`_ below** </div>

In [17]:
#- average number of residents per formal house
f_house = 4
#- average number of residents per informal structure
inf_structure = 3

<div class="alert alert-block alert-warning"><b></b>  
    
**Furthermore:**    
- [social housing](https://en.wikipedia.org/wiki/Public_housing) is tagged `building:residential` with the number of occupants iether the number of informal structure occupants or `building:flats * inf_structure`
- A `social_facility` (carehome, shelter, etc.) harvests the beds `key:value` pair.
- `building:apartments` harvests the `building:flats` `key:value` pair (the number of units) to calculate `*3` people per  
    - ***Student accomodation***:  
>    - University owed: is tagged `building:dormitory` with `residential:university` and harvests the `beds` or `rooms` *'key:value'* pair.
>    - Private for-profit: is tagged `building:residential` or `:dormitory` with `residential:student` and then harvests the `building:flats` or `:rooms` *'key:value'* pair *(the number of units)* to calculate `*1` people per apartment; if `level: > 1` else `*3` people in a house share.  

**The tagging scheme and numbers is based on *how your community is mapped* and local knowledge**
</div>

In [18]:
c = gdf_pop.columns

def pop(row):
    #- formal house
    if row['building'] == 'house' or row['building'] == 'semidetached_house':
        return f_house
    if row['building'] == 'terrace' or row['building'] == 'terraced':
        if 'building:units' in c and row['building:units'] != 0:
            return row['building:units'] * f_house
        else:
            f_house

    #- informal structure (shack)
    if row['building'] == 'cabin':
        return inf_structure
        
    #- in this case social housing
    if row['building'] == 'residential' and 'social_facility' in c and row['social_facility'] is np.nan:
        if row['building:levels'] > 1:
            if 'rooms' in c and row['rooms'] != 0:
                return row['rooms']
            if 'building:flats' in c and row['building:flats'] != 0:
                return row['building:flats'] * inf_structure
        else:
            return inf_structure
    #-- social facility [shelter / carehome]
    if row['building'] == 'residential' and row['social_facility'] is not np.nan:
        if 'building:units' in c and row['building:units'] != 0:
            return row['building:units'] * inf_structure
        else: 
            return row['beds']
                
    #- formal apartment
    if row['building'] == 'apartments':
        return row['building:flats'] * 3
        
    #- private student residence 
    if row['building'] == 'student':
        if row['building:levels'] > 1:
            return row['building:flats']
        else:
            return 3
    # university owned student residence
    if row['building'] == 'dormitory' and row['residential'] == 'university':
        if row['building:levels'] > 1:
            if 'rooms' in c and row['rooms'] != 0:
                return row['rooms']
            if 'beds' in c and row['beds'] != 0:
                return row['beds']
        else:
            return 3

gdf_pop['pop'] = gdf_pop.apply(lambda x: pop(x), axis=1)

est_pop = int(gdf_pop['pop'].sum())
print('The estimated population is:', est_pop)

The estimated population is: 8031


**[Statistics South Africa (STATSA)](https://www.statssa.gov.za) does not typically release official statistics at a suburb level but [City of Cape Town](https://www.capetown.gov.za/Family%20and%20home/education-and-research-materials/data-statistics-and-research/cape-town-census) generously publishes suburb profiles as Open Data.  
These numbers are based on disaggregated [Statistics South Africa (STATSA)](https://www.statssa.gov.za) [Census 2011](https://www.statssa.gov.za/?page_id=3839).** 

**Woodstock 12 656** (6 577: salt river and 9 207: observatory). 

We can calculate the annual population growth rate using the formula for **[Annual population growth](https://databank.worldbank.org/metadataglossary/health-nutrition-and-population-statistics/series/SP.POP.GROW):**

$$r = \frac{\ln{[\frac{End Population}{Start Population}}]}{n} * 100 = \frac{\ln{[\frac{17 062}{9345}}]}{12} * 100   = 2.49\%$$

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    

It is possible to execute the calculation programmatically. **Fill in the relevant variables in the _`cell`_ below** </div>

In [19]:
#- previous population
start_population = 12656

#- period in years from the previous census
years = 12

In [20]:
#-execute
r = (np.log(est_pop/start_population)/years) * 100
print('population growth rate of approximately:', round(r, 2), '%')

population growth rate of approximately: -3.79 %


To conclude; we can project into the future with a very basic formula to estimate the population _x_-years from now:  

$$p  = P_o * (1 + r)^{t} = p = 17062 * (1 + 0.0248)^{10}  = 21 818$$

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    

It is possible to execute the calculation programmatically. **Fill in the variables in the _`cell`_ below** </div>

In [21]:
#- period in years from now
years = 10

In [22]:
#p = est_pop * (1 + (r/100))**years

#print('estimated population', years ,'years from now:', int(p))

#- account for non-residential areas without failure
#- helper function
def safe_population_estimate(est_pop, r, years):
    try:
        p = est_pop * (1 + (r / 100))**years
        return int(p)
    except Exception as e:
        print(f"Population estimate failed: {e}")
        return None  # keeps notebook running

#- execute function
p = safe_population_estimate(est_pop, r, years)

#- shows error and moves on
if p is not None:
    print(f"estimated population {years} years from now: {p}")

estimated population 10 years from now: 5457


<div class="alert alert-block alert-success"><b>1. b) Building Volume Per Capita (BVPC):</b>  
BVPC = total population of a community divided by sum of building volume</div>

In [23]:
#gdf_pop.head(3)

In [24]:
#gdf_pop['area'] = gdf_pop['geometry'].area#\.map(lambda p: p.area)
gdf_pop['area'] = gdf_pop['geometry'].apply(lambda geom: geom.area if geom else 0)
gdf_pop['volume'] = gdf_pop['area'] * gdf_pop['building_height']

#- remove the volume of the ground floor (unoccupied) when building:levels > 7 [this is an arbitrary number based on local knowledge]
#- typically the space is reserved for some other function: retail, etc. 
gdf_pop['volume'] = [
    (row['volume'] - row['area'] * 2.8) if (
        ('social_facility' not in gdf_pop.columns or pd.isna(row.get('social_facility')))
        and row['building:levels'] > 7
        and row['building'] in ['residential', 'apartments', 'student']
    ) else row['volume']
    for _, row in gdf_pop.iterrows()
]

#gdf_pop['bvpc'] =  gdf_pop['volume'] / gdf_pop['pop']
gdf_pop['bvpc'] = np.where(
    gdf_pop['pop'] > 0,
    gdf_pop['volume'] / gdf_pop['pop'],
    np.nan
)

gdf_pop.tail(2)

,id,osm_id,address,building,building:levels,building_height,roof_height,ground_height,plus_code,operator,...,beds,residential,building:use,rooms,building:units,geometry,pop,area,volume,bvpc
1307654032,1371,1307654032,Goldsmith Road Salt River Cape Town,house,1.0,4.1,13.81,9.71,4FRW3F97+XQ7,NaN,...,0,NaN,NaN,0,0,"POLYGON ((265631.975787378 6242700.339320255, ...",4.0,151.280498,620.250042,155.062510
1307654033,1372,1307654033,Goldsmith Road Salt River Cape Town,house,1.0,4.1,13.81,9.71,4FRW3F97+XQX,NaN,...,0,NaN,NaN,0,0,"POLYGON ((265631.975787378 6242700.339320255, ...",4.0,197.021823,807.789474,201.947368


In [25]:
print(gdf_pop['bvpc'].describe())

count     988.000000
mean      109.756705
std        61.754755
min        26.640456
25%        79.875392
50%        91.971120
75%       123.303899
max      1113.202204
Name: bvpc, dtype: float64


In [26]:
bvpc = round(gdf_pop['volume'].sum() / est_pop, 3)

print('Building Volume Per Capita (BVPC):', bvpc)

Building Volume Per Capita (BVPC): 122.544


<div class="alert alert-block alert-info"><b></b>

**This BVPC value is for all buildings with a `population > 0`. Buildings people live in** *(homes)*. 

And we can seperate `building:house` from `building:cabin` and `building:residential` to undertand the differences between ***formal and informal*** housing in this area.
    
**We want to understand the living space *(the cubic-meter BVPC value)* each person has in thier home**
</div>

In [27]:
formal = gdf_pop[gdf_pop["building"].isin(['house', 'semidetached_house', 'terrace', 'terraced', 'apartments'])].copy()
f_pop = int(formal['pop'].sum())
#f_area = formal['area'].mean()

informal = gdf_pop[gdf_pop["building"].isin(['residential', 'cabin'])].copy()
inf_pop = int(informal['pop'].sum())
#inf_area = formal['area'].mean()

#- student
stu = gdf_pop[gdf_pop["building"].isin(['student', 'dormitory'])].copy()
stu_pop = int(stu['pop'].sum())

bvpc_formal = round(formal['volume'].sum() / formal['pop'].sum(), 3)
bvpc_informal = round(informal['volume'].sum() / informal['pop'].sum() if informal['pop'].sum() != 0 else 0, 3)
bvpc_stu = round(stu['volume'].sum() / stu['pop'].sum() if stu['pop'].sum() != 0 else 0, 3)

print('FORMAL: Population: ', f_pop, ' with Building Volume Per Capita (BVPC):', bvpc_formal)
print('')
print('STUDENT RESIDENCE: Population: ', stu_pop, ' with Building Volume Per Capita (BVPC):', bvpc_stu)
print('')
print('INFORMAL: Population: ', inf_pop, ' with Building Volume Per Capita (BVPC)', bvpc_informal)

FORMAL: Population:  6791  with Building Volume Per Capita (BVPC): 118.228

STUDENT RESIDENCE: Population:  1210  with Building Volume Per Capita (BVPC): 140.779

INFORMAL: Population:  30  with Building Volume Per Capita (BVPC) 96.626


<div class="alert alert-block alert-danger"><b>Warning: </b>
    

These are LoD1 3D City Models and works well in these types of areas.  
LoD2 would offer a more representative BVpC [(Ghosh, T; et al. 2020)](https://www.researchgate.net/publication/343185735_Building_Volume_Per_Capita_BVPC_A_Spatially_Explicit_Measure_of_Inequality_Relevant_to_the_SDGs) value; when the complexity of the built environment increases.  

Think about a `house` with living space in the roof structure, so called *'attic living'*, or an `apartment` / `residential` building with different levels, loft apartments and/or units in the turrets of a `building`. 

***consider***: ***[geo3D](https://adriankriger.github.io/geo3D/)*** seperates [building:cabin](https://wiki.openstreetmap.org/wiki/Tag:building%3Dcabin) from `building:residential` to more precisely represent informal structures which typical do not have roof trussess but account for [social housing](https://en.wikipedia.org/wiki/Public_housing) that does</div></div>

## 2. Further examples of Spatial Data Science *(renewable energy)*:

<div class="alert alert-block alert-warning"><b></b>

**Let's attempt to understand the % of homes and population served with renewable energy.**
</div>
    
[**SDG**](https://sdgs.un.org/goals) indicators are typically calculated at **region and national scales**.  
Here, because we are working with highly detailed, local data, we can explore what a [**Tier 3 local indicator**](https://unstats.un.org/sdgs/metadata/) might look like at a ***neighbourhood level***.
<br>

In this section 3. we evaluate [**SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all**](https://sdgs.un.org/goals/goal7) at a community level and estimate the **proportion of residential units and population** that have **direct access to on-site renewable energy infrastructure** *--rooftop photovoltaic panels (PV) and solar water heaters (SWH)*.

- Percentage of **households** served by rooftop renewable energy  
- Percentage of the **population** served by rooftop renewable energy
- And then we go even further to estimate the **Annual Solar Potential in MWh** *(theoretical maximum electricity)* that homes can harvest from the sun over the course of one year.
<div class="alert alert-block alert-info"><b></b> 
    
**PLEASE SUPPLY YOUR OWN [osm.pbf](https://wiki.openstreetmap.org/wiki/PBF_Format).**  

**Either crop an area directly from [OpenStreetMap]() with the ***[official tool](https://www.openstreetmap.org/export#map=3/0.70/22.15)***, select a predefined area [from any number of providers](https://wiki.openstreetmap.org/wiki/Planet.osm), such as ***[Geofabrik](https://download.geofabrik.de)***, or...**
</div>
<div class="alert alert-block alert-success"><b></b>
    
**... download your own. Provincial extracts for South Africa are available here:** *http://download.openstreetmap.fr/extracts/africa/south_africa/*</div>
</div>

In [28]:
# Input OSM PBF file
input_pbf = "../data/CapeTown.osm.pbf"

In [29]:
#- execute function from city3D and return GeoDataFrameLite | home-baked gdf
aoi = city3D.extract_boundaries_by_name(input_pbf, jparams)

geoms = aoi['geometry'].tolist()
#- combine all geometries into a single union
combined_ageom = shapely.unary_union(geoms)  # returns Polygon or MultiPolygon
#- compute bounding box
minx, miny, maxx, maxy = combined_ageom.bounds

#solar = city3D._harvestSolar(input_pbf, jparams, minx, miny, maxx, maxy, crs)
solar = city3D._harvestSolar(input_pbf, minx, miny, maxx, maxy, crs[7:])

if len(solar) < 0:
    print("\033[1m No solar generator features found.\033[0m No rooftop solar are mapped in", focus)

In [30]:
solar.head(2)

,geometry,osm_way_id,other_tags,osm_id,z_order
0,MULTIPOLYGON (((266059.9409322679 6241629.4987...,1175688454,"""generator:method""=>""photovoltaic"",""generator:...",NaN,NaN
1,MULTIPOLYGON (((266059.59639678505 6241642.708...,1175688455,"""generator:method""=>""photovoltaic"",""generator:...",NaN,NaN


In [31]:
# Convert valid strings, ignore None/NaN
def safe_convert(tag_string):
    if isinstance(tag_string, str):
        try:
            # Replace "=>" with ":" and fix newlines
            formatted_string = "{" + tag_string.replace("=>", ":").replace("\n", " ") + "}"
            return json.loads(formatted_string)  # Parse safely
        except json.JSONDecodeError:
            return {}  # Return empty dict on failure
    return {}  # Return empty dict if NaN or None

# Apply conversion function
solar["tags"] = solar["other_tags"].apply(safe_convert)

# Extract values safely - Normalize the 'tags' column to create a new DataFrame
tags_df = pd.json_normalize(solar['tags'])
# Join the new columns back to the original GeoDataFrame
solar = pd.concat([solar, tags_df], axis=1)
# (Optional) Drop the original 'tags' column
solar = solar.drop(columns=['other_tags'])

# Ensure a single 'osm_id' column
if 'osm_id' in solar.columns:
    if 'osm_way_id' in solar.columns:
        solar['osm_id'] = [o if pd.notna(o) else w 
                         for o, w in zip(solar['osm_id'], solar['osm_way_id'])]
        solar = solar.drop(columns=['osm_way_id'])
elif 'osm_way_id' in solar.columns:
    solar = solar.rename(columns={'osm_way_id': 'osm_id'})

#gdf = gdf[gdf.geometry.apply(lambda x: x.within(aoi.unary_union))]
#solar = solar[solar.geometry.apply(lambda x: x.within(shapely.unary_union(aoi.geometry)))]
aoi = aoi.to_crs(crs[7:])
solar = solar[solar.geometry.apply(lambda x: x.within(shapely.unary_union(aoi.geometry)))]
#solar.crs = "EPSG:4326"
#solar = solar.to_crs(crs[7:])

solar.head(2)

,geometry,osm_id,z_order,tags,generator:method,generator:output:electricity,generator:source,generator:type,power,location,generator:output:hot_water
4,POLYGON ((265277.85421120946 6242363.049765201...,1045126952,0.0,"{'generator:method': 'photovoltaic', 'generato...",photovoltaic,yes,solar,solar_photovoltaic_panel,generator,roof,NaN
5,"POLYGON ((265272.3012488816 6242352.679363036,...",1045126953,0.0,"{'generator:method': 'photovoltaic', 'generato...",photovoltaic,yes,solar,solar_photovoltaic_panel,generator,roof,NaN


In [32]:
#- the number of renewable in AREA
solar['generator:method'].value_counts()

generator:method
photovoltaic    121
thermal           4
Name: count, dtype: int64

In [33]:
# join (link) rooftop renewable energy to the appropriate bld
def buildings_with_solar(gdf_buildings, gdf_solar):
    # Prepare output arrays
    solar_ids_per_building = [[] for _ in range(len(gdf_buildings["geometry"]))]
    solar_types_per_building = [[] for _ in range(len(gdf_buildings))]
    #pop = [[] for _ in range(len(gdf_buildings))]
    
    for i, b_geom in enumerate(gdf_buildings["geometry"]):
        for j, s_geom in enumerate(gdf_solar["geometry"]):
            #if b_geom.intersects(s_geom):
            if b_geom.contains(s_geom):
                solar_ids_per_building[i].append(gdf_solar["osm_id"].iloc[j])
                solar_types_per_building[i].append(gdf_solar["generator:method"].iloc[j])
                #pop[i].append(gdf_solar["generator:method"].iloc[j])


    #- keep only unique values
    #unique_methods_per_building = [list(set(lst)) for lst in solar_types_per_building]
    
    gdf_buildings["solar_ids"] = solar_ids_per_building
    gdf_buildings["generator:method"] = solar_types_per_building #unique_methods_per_building
    gdf_buildings["has_solar"] = [len(lst) > 0 for lst in solar_ids_per_building]
    gdf_buildings["solar_ids"] = solar_ids_per_building
    
    return gdf_buildings

blds = buildings_with_solar(gdf_pop, solar)
blds.head(2)

,id,osm_id,address,building,building:levels,building_height,roof_height,ground_height,plus_code,operator,...,rooms,building:units,geometry,pop,area,volume,bvpc,solar_ids,generator:method,has_solar
13665513,4,13665513,Upper East Side 31 Brickfield Road 7935 Cape Town,apartments,8.0,23.7,64.98,41.279999,4FRW3F74+QXM,NaN,...,0,0,POLYGON ((264964.75661551487 6242104.519179016...,750.0,8856.852104,185108.208980,246.810945,[],[],False
13956205,9,13956205,Swift Studio 97 Cecil Road Salt River 7925 Cap...,apartments,3.0,9.7,28.09,18.389999,4FRW3F96+2M3,NaN,...,0,0,"POLYGON ((265381.0475550242 6242446.912382688,...",6.0,688.578683,6679.213224,1113.202204,[],[],False


In [34]:
#--we only want buildings people live in (homes). building=house or =apartment or =residential, etc.

blds = blds[blds["building"].isin(['house', 'semidetached_house', 'terrace', 'terraced', 'apartments', 'residential', 'dormitory', 'cabin', 'student', 'garage'])].copy()

<div class="alert alert-block alert-success"><b>Percentage of households served by rooftop renewable energy</b></div>

$$ \text{\% homes with renewable energy} = \frac{\text{Number of dwellings with mapped solar PV and SWH}}{\text{Total number of dwellings}} \times 100 $$

In [35]:
#- harvest columns
with_solar = sum(blds["has_solar"])
pop = est_pop #gdf["pop"]
total_homes = len(blds)

solHms = round((with_solar / total_homes) * 100, 2)

print('\033[1m Percentage homes, \033[0m in', jparams['FocusArea'],', with rooftop photovoltaic panels (PV) and solar water heaters (SWH):', solHms)

 Percentage homes,  in Salt River , with rooftop photovoltaic panels (PV) and solar water heaters (SWH): 0.4


<div style="text-align: left;"> 
<small> <b>NB:</b> this number includes the <a href="https://wiki.openstreetmap.org/wiki/Tag:building%3Dgarage">OpenStreetMap <code>building=garage</code></a> building type. Go to <code>Cell &plusmn;34</code> (above) to exclude this building type from the estimate.</small>
</div>
<br>
<div class="alert alert-block alert-success"><b>Percentage of population served by rooftop renewable energy</b></div>

$$ \text{\% population with renewable energy} = \frac{\text{Number of residents with mapped solar PV and SWH}}{\text{Estimated population}} \times 100 $$

In [36]:
pop_total = blds["pop"].sum()
pop_solar = blds["pop"][blds["has_solar"]].sum()

solPop = round(pop_solar / pop_total * 100, 2)

print('\033[1m Percentage population, \033[0m in', jparams['FocusArea'],', with rooftop photovoltaic panels (PV) and solar water heaters (SWH):', solPop)

 Percentage population,  in Salt River , with rooftop photovoltaic panels (PV) and solar water heaters (SWH): 0.2


In [37]:
# number of solar renewable on HOMES
blds['generator:method'].explode().value_counts()

generator:method
thermal         4
photovoltaic    2
Name: count, dtype: int64

### 2. ii) Solar potential (MWh)

<div class="alert alert-block alert-warning"><b></b>

In this section, we attempt to understand how much ***'fuel'*** a rooftop can get from the sun.
</div>

<div class="alert alert-block alert-info"><b></b> 
    
**PLEASE SUPPLY YOUR OWN `ghi.tif`. The `ghi.tif` raster comes with the `LTAy_YearlySum.GeoTiff` dataset.  
These can be harvested from the [Global Solar Atlas](https://data360.worldbank.org/en/dataset/WB_SOLAR_ATLAS)** *--a World Bank Group and ESMAP project--* **[download section](https://globalsolaratlas.info/download) and are available for each country on Earth.**  

</div>

We will query the **Global Horizontal Irradiation (GHI**) dataset. This high-resolution raster uses a combination of satellite observations and advanced meteorological models to tell us the *'solar weight'* hitting our neighborhood.

> **Source:** The **Global Solar Atlas GHI Raster** provides a ~20-year historical average of solar radiation. This ensures our communities solar potential is based on **long-term climate trends** rather than a single year of weather.
>   
> **Goal:**
> To calculate the Annual Total GHI (kWh/m2/year). This value tells us the cumulative **'solar pressure'** hitting our rooftops over an entire year, which we then use to calculate **how many Megawatt-hours (MWh) of clean electricity our neighborhood can generate**.
>

In [38]:
#- where is the GHI.tif
raster_file = "../raster/GHI.tif"

In [39]:
#- harvest GHI value from raster

# 1. Setup Raster Access once (Outside the loop)
ds = gdal.Open(raster_file)
gt = ds.GetGeoTransform()
rb = ds.GetRasterBand(1)

# Setup Transformation (OSM/GDF 4326 -> Raster CRS)
target = osr.SpatialReference()
target.ImportFromWkt(ds.GetProjection())
source = osr.SpatialReference()
source.ImportFromEPSG(4326)
transform = osr.CoordinateTransformation(source, target)

def get_ghi_per_building(geom):
    """
    Queries the raster at the specific centroid of the provided geometry.
    Uses the pre-loaded transform and raster band for speed.
    """
    if not geom:
        return 0.0
    
    # Get the specific centroid for this building
    centroid = geom.centroid
    
    # Transform point to Raster's CRS
    # Note: TransformPoint expects (x, y) which is (lon, lat) in 4326
    point = transform.TransformPoint(centroid.x, centroid.y)
    
    # Map Map Coordinates to Pixel Coordinates
    # px = (X_geo - X_origin) / PixelWidth
    # py = (Y_geo - Y_origin) / PixelHeight
    px = int((point[0] - gt[0]) / gt[1])
    py = int((point[1] - gt[3]) / gt[5])
    
    try:
        # Read a 1x1 window at the calculated pixel coordinates
        structval = rb.ReadAsArray(px, py, 1, 1)
        return float(structval[0][0])
    except:
        return 0.0

# 2. Execution
# Ensure your GDF is in 4326 to match the transformer logic
blds = blds.to_crs('EPSG:4326')

# Apply the function to each row's unique geometry
blds['ghi_yearly'] = blds['geometry'].apply(get_ghi_per_building)

print("Annual Average GHI:", round(blds['ghi_yearly'].mean(),2) ,"kWh/m²/year")

Annual Average GHI: 1901.97 kWh/m²/year


<div class="alert alert-block alert-success"><b>Annual Solar Potential (MWh)</b></div>

We use a simplified formula to provide a clear baseline.

$$
\text{Potential (MWh)} = \frac{(\text{Surface Area} \times \text{utilization factor}) \times \text{GHI}_{\text{annual}} \times 0.2}{1000}
$$


<div class="alert alert-block alert-warning"><b></b>

<sub>***Theoretical Framework:** The annual energy output of a photovoltaic system (E) is determined by the product of the total solar resource (GHI), the active area of the array (A), and the system's overall efficiency (η), adjusted by a Performance Ratio (PR) to account for real-world losses. — based on [NREL (2022)](https://joint-research-centre.ec.europa.eu/photovoltaic-geographical-information-system-pvgis_en) & [IEC 61724-1](https://webstore.iec.ch/en/publication/65561). We then adapt this formula and represents a combined value of 25% nominal panel efficiency and a 0.80 Performance Ratio with a **single 0.20 system efficiency value and account for usable area**, a heuristic for gabled roofs.*</sub> 
</div>

<div class="alert alert-block alert-danger"><b>Your Participation! </b>
    
**Fill in the `utilization_factor` below** 
</div>

As a **'rule-of-thumb'** a community with traditional gabled houses: `utilization_factor = 0.4` *(less than half)*, while a high-density suburb with flat-roofed apartments: `utilization_factor = 0.6`

In [40]:
# on average, how much of the roof faces the sun? Adjust based on roof types
utilization_factor = 0.4

In [41]:
# Potential (MWh) = (Area * GHI_Yearly * 0.20) / 1000
blds['solar_mwh'] = ((blds['area'] * utilization_factor) * blds['ghi_yearly'].mean() * 0.20) / 1000
blds.head(2)

,id,osm_id,address,building,building:levels,building_height,roof_height,ground_height,plus_code,operator,...,geometry,pop,area,volume,bvpc,solar_ids,generator:method,has_solar,ghi_yearly,solar_mwh
13665513,4,13665513,Upper East Side 31 Brickfield Road 7935 Cape Town,apartments,8.0,23.7,64.98,41.279999,4FRW3F74+QXM,NaN,...,"POLYGON ((18.457135 -33.9353044, 18.4570908 -3...",750.0,8856.852104,185108.208980,246.810945,[],[],False,1886.0,1347.640175
13956205,9,13956205,Swift Studio 97 Cecil Road Salt River 7925 Cap...,apartments,3.0,9.7,28.09,18.389999,4FRW3F96+2M3,NaN,...,"POLYGON ((18.4617266 -33.93231229999999, 18.46...",6.0,688.578683,6679.213224,1113.202204,[],[],False,1905.0,104.772699


In [42]:
average_solar_potential = blds['solar_mwh'].mean()
print("\033[1mThe average solar potential, per home\033[0m, for", jparams['FocusArea'], "is:", round(average_solar_potential, 2), "MWh/year")

The average solar potential, per home, for Salt River is: 20.57 MWh/year


<div style="text-align: left;"> 
<small> <b>NB:</b> this number includes the <a href="https://wiki.openstreetmap.org/wiki/Tag:building%3Dgarage">OpenStreetMap <code>building=garage</code></a> building type. Go to <code>Cell &plusmn;34</code> to exclude this building type from the estimate.</small>
</div>
<br>
<div class="alert alert-block alert-info"><b></b>
   
What does this **MWh/year** value mean?
<br>
    
To put the value in context, **15 MWh/year:**  
- is enough to provide **100% of the electricity** for 4 to 5 average UK homes *(which use ~3.4 MWh each)* or 1.5 average US homes *(~10.7 MWh each)*
- is enough power to drive an Electric Vehicle for **75,000 kilometers** *--that’s almost two full trips around the Earth*.
- roughly saves **10 metric tons of Carbon Dioxide** from entering the atmosphere.
    
</div>

<div class="alert alert-block alert-danger"><b>Sanity Check!</b>
<br>

- In <code>Cell &plusmn;34</code> we excluded non-residential building types `=office, commercial, retail, warehouse, industrial, etc.` from the analysis.
    
- Cape Town typically yields ~1.6–1.7 MWh **per installed kWp per year**; the higher per-household values reported here reflect **rooftop potential derived from available area** *--that considers a `utilization_factor`*, not a 1 kWp system.

    A 1 kWp solar PV system requires approximately 5–8 m² of panel area (e.g. panels of roughly 1 m × 1.7 m, depending on technology).

    We are NOT asking: How much energy (MWh/year) would a single 1 kWp PV system generate on a roof?  
    We are asking: **How much energy (MWh/year) could these roofs harvest, given their available area and a realistic utilization factor?**

- The **BVPC Warning** applies here too. These are **LoD1 3D City Models**, which represent buildings as simple extrusions.

  LoD2 models *—that capture roof form (e.g. gable, hipped, mansard, domes)—* would provide more representative estimates of both **BVPC and Average Annual Solar Potential**. In such cases, the `utilization_factor` becomes less critical, as usable roof geometry is explicitly modelled. 

</div>

## 3. Interactive Visualization

You might want to create and share an `html` visualization.

<div class="alert alert-block alert-warning"><b> </b>  
    
_In this example we identify building stock by **color** but you are limited only through your imagination and the data you have access too_
</div>

In [43]:
gdf2.crs

<Projected CRS: EPSG:32734>
Name: WGS 84 / UTM zone 34S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 18°E and 24°E, southern hemisphere between 80°S and equator, onshore and offshore. Angola. Botswana. Democratic Republic of the Congo (Zaire). Namibia. South Africa. Zambia.
- bounds: (18.0, -80.0, 24.0, 0.0)
Coordinate Operation:
- name: UTM zone 34S
- method: Transverse Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

<img src="../data/proj.png" alt="proj" width="550" align="right"/>
<br>
<br>

We need a ***Geographic*** Coordinate Reference System.

        <Geographic 2D CRS: EPSG:4326>
        Name: WGS 84
        Axis Info [ellipsoidal]:
        - Lat[north]: Geodetic latitude (degree)
        - Lon[east]: Geodetic longitude (degree)
        Area of Use:
        - name: World.
        - bounds: (-180.0, -90.0, 180.0, 90.0)
        Datum: World Geodetic System 1984 ensemble
        - Ellipsoid: WGS 84
        - Prime Meridian: Greenwich

In [44]:
#- pseudo-3D viz needs geographic coords
gdf2 = gdf2.to_crs(4326)

In [45]:
# -- get the location for 3D viz

# combine all geometries
geom = shapely.unary_union(gdf2['geometry'])
# centroid
xy = (geom.centroid.x, geom.centroid.y)

# bounding box
minx, miny, maxx, maxy = geom.bounds
bbox = [minx, miny, maxx, maxy]

In [46]:
# have a look at the building type and amenities available
gdf2['building'].unique()

array(['office', 'school', 'yes', 'apartments', 'industrial',
       'commercial', 'train_station', 'retail', 'garage', 'workshop',
       'church', 'roof', 'house', 'transportation', 'fire_station',
       'pavilion', 'college', 'dormitory', 'university', 'clubhouse',
       'student', 'warehouse', 'civic', 'parking', 'residential',
       'cinema', 'construction', 'manufacture', 'kindergarten', 'mosque',
       'hall', 'carport', 'terraced', 'guardhouse', 'terrace', 'silo',
       'storage_tank'], dtype=object)

<div class="alert alert-block alert-success"><b>Building Stock:</b> To differentiate a school, housing, retail, healthcare and community focused facilities (library, municipal office, community centre) we color the buildings - we harvest the osm tags [amenity and building type] directly.</div>

In [47]:
#-- colour the building stock based on building:type

## while we can color with a built-in pydeck function
#color_lookup = pdk.data_utils.assign_random_colors(build_df['building'])
 # Assign a color
#build_df['color'] = build_df.apply(lambda row: color_lookup.get(row['building']), axis=1)

## we define specific colors
def color(bld):
    #- formal house
    if bld == 'house' or bld == 'semidetached_house':
        return [255, 255, 204]        #-grey
    #- informal structure / social housing
    if bld == 'residential' or bld == 'dormitory' or bld == 'cabin':
        return [119, 3, 252]          #-purple
    if bld == 'apartments':
        return [252, 194, 3]          #-orange 
    if bld == 'garage' or bld == 'parking':
        return [3, 132, 252]          #-blue        
    if bld == 'retail' or bld == 'supermarket':
        return [253, 141, 60]
    if bld == 'office' or bld == 'commercial':
        return [185, 206, 37]
    if bld == 'school' or bld == 'kindergarten' or bld == 'university' or bld == 'college':
        return [128, 0, 38]
    if bld == 'clinic' or bld == 'doctors' or bld == 'hospital':
        return [89, 182, 178]
    if bld == 'community_centre' or bld == 'service' or bld ==  'post_office' or bld == 'hall' \
    or bld ==  'townhall' or bld == 'police' or bld ==  'library':
        return [181, 182, 89]
    if bld == 'warehouse' or bld == 'industrial':
        return [193, 255, 193]
    if bld == 'restaurant' or bld == 'hotel':
        return [139, 117, 0]
    if bld == 'place_of_worship' or bld == 'church' or bld == 'mosque':
        return [225, 225, 51]
    else:
        return [255, 255, 204]

gdf2["fill_color"] = gdf2['building'].apply(lambda x: color(x))

In [48]:
#- look
gdf2.head(2)

,id,osm_id,address,building,building:levels,building_height,roof_height,ground_height,plus_code,operator,...,amenity,social_facility,bottom_roof_height,beds,residential,building:use,rooms,building:units,geometry,fill_color
6383946,0,6383946,Western Cape Metrorail - Infrastructure Buildi...,office,1.0,4.1,8.21,4.110000,4FRW3FCF+946,NaN,...,NaN,NaN,NaN,0,NaN,NaN,0,0,"POLYGON ((18.4723433 -33.928894799999995, 18.4...","[185, 206, 37]"
13328172,1,13328172,NaN,school,2.0,6.9,23.93,17.030001,4FRW3F96+6MG,NaN,...,NaN,NaN,NaN,0,NaN,NaN,0,0,"POLYGON ((18.4615816 -33.9317448, 18.4615639 -...","[128, 0, 38]"


<div class="alert alert-block alert-success"><b>Additional Features:</b> 

Because we do NOT harvest an online basemap for the pseudo-3D visualisation we need to create our own.  
We add features to our home-baked visualization; *namely: roads and parks. We get this from the [OpenStreetMap](https://en.wikipedia.org/wiki/OpenStreetMap) as well.*.</div>

In [49]:
#- harvest ROADS 
gdf_roads = city3D.process_osm_geoms(
    "/vsimem/roads.geojson", 
    input_pbf,
    "lines", 
    "highway IS NOT NULL", 
    combined_ageom
)
gdf_roads.head(2)

,geometry,highway,name,osm_id,z_order,lanes,lanes:backward,lanes:forward,maxspeed,surface,...,horse_scale,crossing:markings,placement:backward,cycleway:right,parking:left,parking:left:orientation,crossing:island,tactile_paving,indoor,level
0,"LINESTRING (18.4852221 -33.9278552, 18.4852757...",tertiary,Alexandra Road,4247137,4,3,2,1,60,asphalt,...,,,,,,,,,,
1,"LINESTRING (18.4444294 -33.9327723, 18.4445549...",footway,Taliep Petersen Bridge,4256654,20,,,,,,...,,,,,,,,,,


In [50]:
#- LEISURE (parks, gardens, nature reserves)
gdf_green = city3D.process_osm_geoms(
    "/vsimem/green.geojson", 
    input_pbf,
    "multipolygons", 
    #"leisure IN ('park', garden', 'nature_reserve', 'common', 'grass', 'pitch')", 
    "leisure IS NOT NULL", 
    combined_ageom
)

green_types = ['park', 'garden', 'nature_reserve', 'common', 'grass', 'pitch']

# This checks: "Is the value in the leisure column one of the items in my list?"
gdf_green = gdf_green[gdf_green['leisure'].isin(green_types)].copy()
gdf_green.head(2)

ERROR 1: Non closed ring detected.
ERROR 1: Non closed ring detected.
ERROR 1: Non closed ring detected.
ERROR 1: Non closed ring detected.
ERROR 1: Non closed ring detected.
ERROR 1: Non closed ring detected.
ERROR 1: Non closed ring detected.
ERROR 1: Non closed ring detected.
ERROR 1: Non closed ring detected.


,building,geometry,landuse,leisure,name,natural,osm_id,sport,type,addr:city,...,garden:type,hoops,location,phone,opening_hours,fee,garden:style,playground,club,area:highway
2,,"MULTIPOLYGON (((18.4752017 -33.9364824, 18.475...",,park,Two Rivers Park,,11201474,,multipolygon,,...,,,,,,,,,,
6,,"MULTIPOLYGON (((18.4706624 -33.9413838, 18.470...",,pitch,,,4816696,soccer,,,...,,,,,,,,,,


In [51]:
file = '../result/interactiveAlt.html' # will name and save html here

city3D.create_maplibre_3Dviz(
    file,
    buildings_gdf=gdf2,
    roads_gdf=gdf_roads,   
    green_gdf=gdf_green,
    center=xy,
    offline=True           # Toggle for Full Bakery mode   
)
    
#- will open in a new browser window. Jupyter security restrictions prevent opening here. 
webbrowser.open('file://' + os.path.realpath(file))

True

**on a laptop without a mouse:**

- `trackpad left-click drag-left` and `-right`;
- `Ctrl left-click drag-up`, `-down`, `-left` and `-right` to rotate and so-on and
- `+` next to Backspace zoom-in and `-` next to `+` zoom-out.

**Now you do your community.** ~ If your area needs [OpenStreetMap](https://en.wikipedia.org/wiki/OpenStreetMap)  data and you want to contribute please follow the [Guide](https://wiki.openstreetmap.org/wiki/Beginners%27_guide).

## 4. Possible Secondary and Tertiary level *conversations starters*:

<div class="alert alert-block alert-success"><b>communicate and exchange ideas and understanding</b></div>

| **Topic**                                | **Secondary Level Questions**                                                                                                                                                                                   | **Tertiary Level Questions**                                                                                                                                                                                                                   |
|------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Basic Understanding and Observations** | - What types of buildings are most common in the area (houses, apartments, retail, etc.)?<br>- Can you identify any patterns in the distribution of different types of buildings (e.g., are retail stores concentrated in certain areas)? | - How does the building stock composition (e.g., ratio of houses) correlate with the population? *demographics (e.g., age distribution, household size) for the area will strengthen the analysis!* <br>- Analyze the relationship between building density and population. What urban planning theories can explain this relationship? |
| **Spatial Relationships and Impacts**    | - How does the location of residential areas compare to the location of retail and commercial areas?<br>- What impact might the density and distribution of buildings have on local traffic and transportation?<br>- How might the population distribution affect the demand for local services such as schools, hospitals, and parks? | - Evaluate the accessibility of essential services (e.g., healthcare, education) in relation to the population and building types.<br>- Assess the potential social and economic impacts of a proposed new residential or commercial development in the area.                  |
| **Socioeconomic and Environmental Considerations** | - Are there any correlations between the types of housing available and the household size? *additional demographics (e.g., income level) for the area will strengthen the analysis!*<br>- How might the current building stock and population influence the local economy? *demographics (e.g., age distribution, household size) for the area will strengthen the analysis!*<br>- What are some potential environmental impacts of the current building distribution, such as green space availability or pollution levels? | - How does the current building stock support or hinder sustainable development goals (e.g., energy efficiency, reduced carbon footprint)?<br>- What strategies could be implemented to increase the resilience of the community to environmental or economic changes?                       |
| **Future Planning and Development**      | - Based on the current building stock and population metrics, what areas might benefit from additional housing or commercial development?<br>- How could urban planners use this information to improve the quality of life in the area?<br>- What changes would you recommend to better balance residential, commercial, and recreational spaces? | - How might different zoning regulations impact the distribution of residential, commercial, and industrial buildings in the future?<br>- Propose urban design solutions that could improve the sustainability and livability of the area, considering both current metrics and future projections. |
| **Quantitative and Qualitative Research** | |- Design a research study to investigate the impact of building type diversity on community wellbeing. What methodologies would you use?<br>- Analyze historical data to understand trends in building development and population growth. How have these trends shaped the current urban landscape?<br>- Conduct a SWOT analysis (Strengths, Weaknesses, Opportunities, Threats) of the area based on the building stock and population metrics. |